In [0]:
spark.conf.set("spark.sql.adaptive.enable", "false")
spark.conf.set("spark.sql.adaptive.dynamicPartitionPruning.enabled", "false")
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)

In [0]:
ext_loc='abfss://devcontainer@venubtc72.dfs.core.windows.net'
csv_file=ext_loc+'/inbound/Listings.csv'
from pyspark.sql.functions import col, lit

In [0]:
df = spark.read.options(header='true', inferSchema='true').csv(csv_file)
df2=df.limit(5)
df2.display()


In [0]:
df2.filter(df.PostalCode.isNull()).count()

In [0]:
df2.printSchema()

In [0]:
parquet_loc=ext_loc+"/parquets"

df2.write \
    .format("parquet") \
    .mode("append") \
    .partitionBy("PostalCode") \
    .option("path", parquet_loc) \
    .save()

In [0]:
df2.rdd.getNumPartitions()

 Non partition data 

In [0]:
dbutils.fs.rm(
    "abfss://devcontainer@venubtc72.dfs.core.windows.net/non_partition/",
    True
)

In [0]:
non_partition_loc=ext_loc+"/non_partition"

In [0]:
df2.printSchema()

In [0]:

dbutils.fs.rm(non_partition_loc, True)

df2.coalesce(4).write \
    .mode("append") \
    .parquet(non_partition_loc)

In [0]:
non_partition_loc=ext_loc+"/non_partition"

df2.coalesce(4).write \
    .format("parquet") \
    .mode("overwrite") \
    .option("path", non_partition_loc) \
    .save()

In [0]:
df2 = spark.read.format('parquet')\
      .load(non_parquet_loc)
df2.count()

In [0]:
df2.write \
    .format("parquet") \
    .mode("overwrite") \
    .partitionBy("PostalCode") \
    .option("path", parquet_loc) \
    .save()